# 02 - Stationarity Tests - Spain CPI

Formal determination of the integration order of the Spain CPI series.

**Tests applied:**
- **ADF** (Augmented Dickey-Fuller) - H0: unit root (non-stationary)
- **KPSS** - H0: stationary (complement to ADF)
- **Phillips-Perron** - H0: unit root (robust to heteroscedasticity)

Applied to: level series, first difference, and seasonal difference (lag 12).

**Input:** `data/processed/ipc_spain_index.parquet`  
**Determines:** d and D parameters for ARIMA/SARIMA

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, kpss
from arch.unitroot import PhillipsPerron

ROOT = Path('..').resolve()         # tfg-forecasting/
MONOREPO = ROOT.parent              # tfg-ipc-mcp/
sys.path.insert(0, str(MONOREPO))

from shared.constants import DATE_TRAIN_END

plt.rcParams.update({"figure.figsize": (14, 4), "axes.grid": True, "grid.alpha": 0.3})

In [ ]:
df = pd.read_parquet(ROOT / "data" / "processed" / "ipc_spain_index.parquet")

# Use only the train set for tests (avoid data leakage)
train = df.loc[:DATE_TRAIN_END]
y = train["indice_general"]
print(f"Train: {y.index.min().date()} -> {y.index.max().date()} ({len(y)} obs)")

## 1. Auxiliary Test Functions

In [ ]:
def run_adf(series, name=""):
    """Augmented Dickey-Fuller. H0: unit root."""
    result = adfuller(series.dropna(), autolag="AIC")
    stat, pval, lags, nobs = result[0], result[1], result[2], result[3]
    cv = result[4]
    reject = pval < 0.05
    return {
        "test": "ADF", "series": name,
        "statistic": round(stat, 4), "p_value": round(pval, 4),
        "lags": lags, "nobs": nobs,
        "cv_1%": round(cv["1%"], 4), "cv_5%": round(cv["5%"], 4), "cv_10%": round(cv["10%"], 4),
        "reject_H0": reject,
        "conclusion": "Stationary" if reject else "Non-stationary",
    }


def run_kpss(series, name="", regression="c"):
    """KPSS. H0: stationary. regression='c' (level) or 'ct' (trend)."""
    stat, pval, lags, cv = kpss(series.dropna(), regression=regression, nlags="auto")
    reject = pval < 0.05
    return {
        "test": f"KPSS({regression})", "series": name,
        "statistic": round(stat, 4), "p_value": round(pval, 4),
        "lags": lags,
        "cv_1%": round(cv["1%"], 4), "cv_5%": round(cv["5%"], 4), "cv_10%": round(cv["10%"], 4),
        "reject_H0": reject,
        "conclusion": "Non-stationary" if reject else "Stationary",
    }


def run_pp(series, name=""):
    """Phillips-Perron. H0: unit root."""
    pp = PhillipsPerron(series.dropna())
    reject = pp.pvalue < 0.05
    return {
        "test": "Phillips-Perron", "series": name,
        "statistic": round(pp.stat, 4), "p_value": round(pp.pvalue, 4),
        "lags": pp.lags,
        "reject_H0": reject,
        "conclusion": "Stationary" if reject else "Non-stationary",
    }


def battery(series, name):
    """Run all three tests on a series."""
    return [
        run_adf(series, name),
        run_kpss(series, name, regression="c"),
        run_kpss(series, name, regression="ct"),
        run_pp(series, name),
    ]

## 2. Tests on the Level Series

In [ ]:
results = battery(y, "CPI level")
df_nivel = pd.DataFrame(results)
df_nivel

## 3. First Difference (d=1)

In [ ]:
y_diff1 = y.diff().dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(y.index, y, linewidth=1)
axes[0].set_title("Level Series")
axes[1].plot(y_diff1.index, y_diff1, linewidth=1, color="#d62728")
axes[1].set_title("First Difference (d=1)")
axes[1].axhline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
results += battery(y_diff1, "CPI diff(1)")
df_diff1 = pd.DataFrame(battery(y_diff1, "CPI diff(1)"))
df_diff1

## 4. Seasonal Difference (D=1, lag 12)

In [ ]:
y_diff12 = y.diff(12).dropna()

fig, ax = plt.subplots()
ax.plot(y_diff12.index, y_diff12, linewidth=1, color="#2ca02c")
ax.set_title("Seasonal Difference (lag 12)")
ax.axhline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
results += battery(y_diff12, "CPI diff(12)")
df_diff12 = pd.DataFrame(battery(y_diff12, "CPI diff(12)"))
df_diff12

## 5. Double Difference: d=1 + D=1

In [ ]:
y_diff1_12 = y.diff().diff(12).dropna()

fig, ax = plt.subplots()
ax.plot(y_diff1_12.index, y_diff1_12, linewidth=1, color="#9467bd")
ax.set_title("Double Difference: diff(1) + diff(12)")
ax.axhline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
results += battery(y_diff1_12, "CPI diff(1)+diff(12)")
df_double = pd.DataFrame(battery(y_diff1_12, "CPI diff(1)+diff(12)"))
df_double

## 6. Summary Table

In [ ]:
summary = pd.DataFrame(results)
cols_show = ["test", "series", "statistic", "p_value", "conclusion"]
summary[cols_show].style.applymap(
    lambda v: "background-color: #d4edda" if v == "Stationary" else
              "background-color: #f8d7da" if v == "Non-stationary" else "",
    subset=["conclusion"]
)

## 7. Conclusion

**Joint interpretation ADF + KPSS + PP:**

| Transformation | ADF | KPSS | PP | Decision |
|---|---|---|---|---|
| Level | Fails to reject H0 | Rejects H0 | Fails to reject H0 | Non-stationary → differencing required |
| diff(1) | Rejects H0 | Fails to reject H0 | Rejects H0 | Stationary → **d=1** confirmed |
| diff(12) | Check | Check | Check | Determines whether D=1 suffices |
| diff(1)+diff(12) | Rejects H0 | Fails to reject H0 | Rejects H0 | Confirms d=1, D=1 for SARIMA |

*Note: fill the table with actual values after running the tests.*